In [1]:
# %% [markdown]
# # Memoria de GW (solo isotonic) usando la mejor cuenca por pozo
# - Entrada: `r2_LR_8basins_best_level.csv` (una fila por pozo, combinación y tipo de lag; ya trae el mejor nivel).
# - Memoria = primer rezago donde la curva isotónica de R² vs rezagos alcanza `perc_max_r2` × R² máximo.
# - No se filtran pozos por R².
# - Se exporta el nivel, `HYBAS_ID`, `UP_AREA`, cuencas aguas arriba y cobertura de P de la cuenca elegida.

# %%
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.isotonic import IsotonicRegression

# === Parámetros ===
best_file = Path('csv/r2_LR_8basins_best_level.csv')     # salida del script de regresión (ajustá la ruta)
data_dir = Path('../../data/ms_data/')                   # acá están los wells_meta_levXX.csv
out_results = Path('csv_memory')
out_figs = Path('figs_gw_memory')
out_results.mkdir(parents=True, exist_ok=True)

selected_lag_type_gw = 'type2_incr4'
perc_max_r2 = 0.9          # memoria = primer rezago con R² >= 0.9 * R² máximo del pozo
make_plots = False         # True: una figura por pozo y variante (R² vs rezagos + ajuste isotónico)

combos = [                 # (target, predictor, variante)
    ('gw_an',    'pr_an',    'raw'),   # con tendencia
    ('gw_an_dt', 'pr_an_dt', 'dt'),    # sin tendencia
]


# %% [markdown]
# ## Función de memoria (isotonic)

# %%
def compute_memory_isotonic(x, y, perc_max_r2=0.9):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[ok], y[ok]
    if len(x) == 0:
        return np.nan, None, None
    idx = np.argsort(x)
    x, y = x[idx], y[idx]

    y_iso = IsotonicRegression(increasing=True).fit_transform(x, y)
    x_dense = np.linspace(x.min(), x.max(), 500)
    y_dense = np.interp(x_dense, x, y_iso)

    target = perc_max_r2 * np.nanmax(y)
    above = y_dense >= target
    mem = x_dense[np.argmax(above)] if above.any() else np.nan
    return mem, x_dense, y_dense


# Fin de cada ventana de rezagos: lag_ranges = "[(0, 4), (4, 8), ...]" -> 4, 8, ...
pattern = r"\(?\s*[-\d.]+\s*,\s*([-\d.]+)\s*\)?"


def lags_from_ranges(lr):
    vals = [float(v) for v in re.findall(pattern, str(lr))][:6]
    return vals + [np.nan] * (6 - len(vals))


# %% [markdown]
# ## Cargar mejores cuencas y metadatos

# %%
best = pd.read_csv(best_file, dtype={'well_id': str})
best = best[best['lag_type'] == selected_lag_type_gw].copy()

meta = {}
for lvl in sorted(best['nivel'].unique()):
    m = pd.read_csv(data_dir / f'wells_meta_lev{int(lvl):02d}.csv', dtype={'cr2sub_id': str})
    m['n_cuencas_aguas_arriba'] = m['HYBAS_ID_upstream'].astype(str).str.count(';')   # sin contar la propia
    meta[int(lvl)] = m.drop_duplicates('cr2sub_id').set_index('cr2sub_id')

# %% [markdown]
# ## Memoria por pozo y variante (sin filtro de R²)

# %%
rows = {}
for target, predictor, variant in combos:
    d = best[(best['target'] == target) & (best['predictor'] == predictor)]

    for _, r in d.iterrows():
        well, lvl = r['well_id'], int(r['nivel'])
        x = lags_from_ranges(r['lag_ranges'])
        y = [r[f'r2_pred{i}'] for i in range(1, 7)]
        mem, xd, yd = compute_memory_isotonic(x, y, perc_max_r2)

        row = rows.setdefault(well, {'well_id': well})
        row[f'nivel_{variant}'] = lvl
        row[f'HYBAS_ID_{variant}'] = r['HYBAS_ID']
        row[f'UP_AREA_{variant}'] = r['UP_AREA']
        if well in meta[lvl].index:
            mm = meta[lvl].loc[well]
            row[f'n_cuencas_aguas_arriba_{variant}'] = mm['n_cuencas_aguas_arriba']
            row[f'cobertura_P_{variant}'] = mm.get('cobertura_P', np.nan)
            row[f'HYBAS_ID_upstream_{variant}'] = mm['HYBAS_ID_upstream']
        row[f'r2_{variant}'] = r['r2']
        row[f'memory_iso_{variant}'] = mem

        if make_plots and xd is not None:
            import matplotlib.pyplot as plt
            out_figs.mkdir(parents=True, exist_ok=True)
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.plot(x, y, 'o', color='purple', label=selected_lag_type_gw)
            ax.plot(xd, yd, '--', color='black', lw=1, label='Isotonic fit')
            if not np.isnan(mem):
                ax.axvline(mem, color='black', lw=1, label=f'memory = {mem:.0f} mo')
            ax.set_title(f"Well {well} [{variant}] nivel {lvl}, UP_AREA={r['UP_AREA']:.0f} km², R²={r['r2']:.2f}",
                         fontsize=9, loc='left')
            ax.set_xlabel("P lags (months)"); ax.set_ylabel("R²")
            ax.grid(True, ls='--', alpha=.5); ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(out_figs / f"gw_memory_{well}_{variant}.png", bbox_inches='tight')
            plt.close(fig)

df_mem = pd.DataFrame(list(rows.values()))

cols = ['well_id']
for v in ['raw', 'dt']:
    cols += [f'{c}_{v}' for c in ['nivel', 'HYBAS_ID', 'UP_AREA', 'n_cuencas_aguas_arriba',
                                  'cobertura_P', 'r2', 'memory_iso', 'HYBAS_ID_upstream']]
df_mem = df_mem.reindex(columns=cols)

out_csv = out_results / 'gw_memory_wide_best_basin.csv'
df_mem.to_csv(out_csv, index=False)
print(f"Saved {len(df_mem)} wells to {out_csv}")
print("\nLR models Finished! \nFinito! \nTerminado!\n")

Saved 503 wells to csv_memory\gw_memory_wide_best_basin.csv

LR models Finished! 
Finito! 
Terminado!

